# Epoch budget: the core, or the epochs?

Produces Figure 14 of the paper (Appendix H.4, "Impact of stage 2 training and
epoch budget"): the two-stage fit of SCORE against a stage-1-only model at the
same total epoch budget.

**The question.** SCORE's two-stage fit splits its epoch budget between fitting
the marginals (stage 1) and fitting the correlation core on top of them
(stage 2). Every gain it reports is therefore confounded with *how the budget
was spent*: a reader can object that the same total number of epochs poured
entirely into the marginals would have done as well. This notebook is the
measurement that separates the two.

At each rung of a budget ladder, two arms are trained at the **same total epoch
budget** $E$ and scored on the test set:

| arm | what it is | driven by |
|---|---|---|
| `marginal` | a diagonal Gaussian on the marginal CRPS for the full $E$ epochs | `trainer.max_epochs` |
| `ours` | SCORE: the same backbone, two-stage fit, $3E/4$ marginals $+\ E/4$ core | `stage{1,2}_max_epochs` |

Identical total epochs, identical backbone, identical optimizer and LR recipe,
and the *same* stage-1 objective on both sides -- the control's CRPS is exactly
what SCORE's stage 1 optimizes. So a difference between the curves is
the core, not the training length.

The rung is a **fraction of each experiment's own configured budget**, never an
absolute epoch count, so the $1.0$ rung is the main-experiment configuration by
construction (EUPPBench $40 \to 30+10$, time series $100 \to 75+25$).

**How to read it.** The joint scores (log score, ES, KS, VS) are where the core
can pay off; the marginal ones (CRPS, MSE) are where it cannot, since SCORE
reaches its marginals with a quarter fewer epochs and then freezes them. A
gain on the joint scores at no cost on the marginal ones is the strong result. A
"gain" that is really a marginal-fit difference shows up as movement in the CRPS
panel.

## Reading the results

Everything comes from the jsonl `src/evaluation/epoch_budget.py` appends to,
one line per (dataset, arm, budget, seed) with the full test block. Unlike the
rest of the paper's figures this does **not** read the checkpoint tree: these
runs are deliberately never synced as checkpoints, because a config slug
carries no epoch budget, so every rung of one arm would collapse onto one
path and overwrite each other *and* the main experiment's `seed<N>` folders.

Rows are **deduplicated on the cell key** on the way in. The file is
append-only, may be merged from several machines, and a restarted sweep
re-checks the file rather than locking it, so duplicate cells are expected.
Deduplication is lossless -- training is deterministic given the seed, so
duplicate rows are identical.

In [ ]:
import json
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


def _project_root(start=None):
    """Nearest ancestor with a `.project-root` marker -- same rule plot_config uses.

    Lets the notebook be run from the repository root or from `notebooks/`
    without the import below or the input path depending on which.
    """
    start = Path.cwd() if start is None else Path(start)
    for parent in [start, *start.parents]:
        if (parent / ".project-root").exists():
            return parent
    return start


ROOT = _project_root()
NB_DIR = ROOT / "notebooks"
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import plot_config as pc
import make_results_tables as mrt          # metric columns + d, shared with the tables

CFG = pc.apply_style()

#: The ablation's own record, written by `src/evaluation/epoch_budget.py`.
SCORES = ROOT / "logs" / "epoch_budget" / "scores.jsonl"

#: The results tree. Only the export at the bottom writes here; nothing above
#: reads it.
from src.utils.paths import configured_path
SHARED = configured_path("results_dir")

#: Fields whose tuple identifies one cell of the study -- the dedup key, and the
#: same one the driver's skip check uses.
KEY_FIELDS = ("schema_version", "experiment", "dataset", "mode", "horizon",
              "arm", "budget", "seed")

#: One panel per metric, in the column order of the paper's results tables
#: (`make_results_tables.COLUMNS`) and with its labels. All are lower-is-better.
#: `nll_per_d` is the log score divided by the event dimension, as in the tables.
METRICS = [
    ("mse", "MSE"),
    ("crps", "CRPS"),
    ("energy_score", "energy score"),
    ("variogram_score", "variogram score"),
    ("kernel_score", "kernel score"),
    ("nll_per_d", r"log score / $d$"),
]

#: The two arms. Colour carries the *fitting objective* throughout the paper
#: (plot_config.COLORS): SCORE's core is kernel-score fitted, and the
#: control is the reference model every claim is made against.
ARMS = [
    dict(key="marginal", label="marginal (CRPS), full budget",
         color=pc.COLORS["truth"], marker="s"),
    dict(key="ours", label="SCORE (staged), 3:1 split",
         color=pc.COLORS["kernel_score"], marker="o"),
]

#: Raw test scores every row carries. Non-finite entries (a Dirac's log score)
#: are stored as null by the driver and read back as NaN.
_RAW_METRICS = ("nll", "crps", "energy_score", "kernel_score",
                "variogram_score", "mse", "mae")

In [ ]:
def read_scores(path=SCORES):
    """One row per trained cell, deduplicated on `KEY_FIELDS`.

    Returns a frame with one row per (task, arm, budget, seed). `task` is the
    tuple that identifies a panel; `frac` is the x axis. The first occurrence of
    a key wins, matching the driver's "already present" semantics.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"{path} does not exist -- run `bash runs/epoch_budget.sh` first "
            "(or `sbatch runs/epoch_budget.sh` on the cluster)."
        )
    seen, rows, duplicates, unreadable = set(), [], 0, 0
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            r = json.loads(line)
        except json.JSONDecodeError:
            # A sweep killed mid-write leaves a partial last line; the rows before it
            # are still valid.
            unreadable += 1
            continue
        key = tuple(r.get(f) for f in KEY_FIELDS)
        if key in seen:
            duplicates += 1
            continue
        seen.add(key)
        row = {k: r.get(k) for k in
               ("experiment", "dataset", "mode", "horizon", "arm", "frac",
                "budget", "stage1_epochs", "stage2_epochs", "seed", "slug",
                "params", "best_epoch", "wall_s")}
        row.update({m: r.get("test", {}).get(m) for m in _RAW_METRICS})
        rows.append(row)

    df = pd.DataFrame(rows)
    print(f"{len(df)} cells from {path}"
          + (f"; {duplicates} duplicate row(s) dropped" if duplicates else "")
          + (f"; {unreadable} unreadable line(s)" if unreadable else ""))
    if df.empty:
        return df
    # A task with neither a mode nor a horizon (EUPPBench) stores both as null,
    # and pandas then reads `mode` back as NaN and `horizon` as a float. NaN is
    # truthy, so string joins, `f"T{horizon}"` labels, the dimension lookup and
    # `int(row["horizon"])` in the export would all go wrong. Normalise once here,
    # into `object` columns (a plain list of ints and Nones is re-inferred as
    # float64 with NaN), and keep `horizon` an int so ETT tasks never become `T96.0`.
    df["mode"] = df["mode"].astype(object).where(df["mode"].notna(), None)
    df["horizon"] = pd.Series(
        [None if pd.isna(h) else int(h) for h in df["horizon"]],
        index=df.index, dtype=object,
    )
    # `task` is the *name*, not the (experiment, dataset, mode, horizon) tuple:
    # it becomes an index level below, and a tuple-valued level makes
    # `agg.loc[(task, arm)]` ambiguous -- pandas reads the outer tuple as a
    # multi-level key and raises. The four components stay as columns, which is
    # all `out_dir_for` needs.
    df["task"] = [task_name(d_, m, h) for d_, m, h
                  in zip(df["dataset"], df["mode"], df["horizon"])]
    df["d"] = [dimension(d_, m, h) for d_, m, h
               in zip(df["dataset"], df["mode"], df["horizon"])]
    # The per-coordinate log score, the last panel. NaN where d is unknown, so
    # the panel is empty rather than showing a raw log score in its place.
    df["nll_per_d"] = df["nll"] / df["d"]
    return df


def dimension(dataset, mode, horizon):
    """Event dimension `d` of a task, from `make_results_tables.DIMENSION`.

    The same `(dataset, features)` key the paper's tables use, so `log score / d`
    means the same number in both. `DIMENSION` is keyed at the horizon the
    campaign reports (`pred_len = 96` for the ETT sets, which is what this sweep
    trains); a rung at another horizon has no entry and gets `None`, which
    empties the log-score panel rather than mislabelling it.
    """
    features = {"univariate": "S", "multivariate": "M"}.get(mode or "")
    d = mrt.DIMENSION.get((dataset, features))
    if d is None or horizon in (None, 96):
        return d
    return None


def task_name(dataset, mode, horizon):
    """The task as its path in the shared tree, e.g. `etth1/univariate/T96`."""
    parts = [dataset, mode, f"T{horizon}" if horizon else None]
    return "/".join(p for p in parts if p)


df = read_scores()

The coverage table is the honest status of the figure below: every cell it
needs, and how many seeds back it. The sweep is 2 arms x 5 seeds at every rung of the data
set's ladder -- ten for the ETT sets (`frac` 0.1 ... 1.0), nine for euppbench,
which cannot train the 0.1 rung -- so a complete task reads `5` everywhere. Anything less becomes a wider
error bar, never a quietly interpolated point.

In [ ]:
def coverage(df):
    """Seeds per (task, arm, budget), as a table with the budgets as columns."""
    if df.empty:
        print("no rows")
        return df
    return (df.groupby(["task", "arm", "frac"])["seed"].nunique()
              .unstack("frac").fillna(0).astype(int))


coverage(df)

## Aggregation

For data set $t$, arm $a$ and rung $f$, over the seeds $s$ that ran:

$$
\bar v_t(a, f) = \operatorname*{mean}_s v_{t,s}(a,f),
\qquad
\mathrm{sd}_t(a,f) = \sqrt{\tfrac{1}{n-1}\textstyle\sum_s \big(v_{t,s}(a,f) - \bar v_t(a,f)\big)^2}.
$$

The sample deviation (not the population one) is what a rerun would see, and it
is what the error bars show. It is undefined at $n = 1$ and left as `NaN` there,
so a single-seed rung draws a point with no bar rather than one of zero width.

In [ ]:
def aggregate(df, metrics=None):
    """Mean, sample sd and seed count per (task, arm, frac) and metric."""
    metrics = [m for m, _ in METRICS] if metrics is None else metrics
    out = (df.groupby(["task", "arm", "frac", "budget"])[metrics]
             .agg(["mean", "std", "count"]))
    return out.sort_index()


agg = aggregate(df)
agg.head(10)

## The figure

One row per data set, one panel per metric, and in each panel the two arms
against the epoch budget as a fraction of the configured one -- the budget is
the shared x axis, labelled once under the figure. The shaded band is
$\pm 1$ sample standard deviation across seeds: with five seeds a quantile band would be the 2nd and 4th order
statistic and would jump from rung to rung on noise, and a min-max envelope
would *narrow* wherever a rung is still missing seeds. A rung with a single seed
has no deviation, so its band pinches to the line and only the marker shows.

Every panel is lower-is-better, and the two arms sit at the *same* x for every
rung by construction -- that is the whole design: at a given x the two curves
cost the same number of epochs, so the vertical gap between them is what the
correlation core buys (or costs) at that budget.

In [ ]:
def tasks_of(agg):
    """The data sets present, in a stable order."""
    return sorted(agg.index.get_level_values("task").unique())


#: Row label on the y axis: the data set name alone, which is all that
#: distinguishes the rows of this figure. Overridden where the bare name reads
#: badly. NOTE this drops the mode and the horizon -- if one data set ever
#: appears twice in the same figure (univariate *and* multivariate, or two
#: horizons) the labels collide and the dropped components have to come back.
AXIS_LABEL = {"euppbench": "EUPPBench"}

#: Tick format on the score axes: three fixed decimals on every panel, so all
#: label columns have the same width. Significant-digit formats give ragged
#: widths, and two decimals collapse the panels whose arms differ only in the
#: third decimal. A few panels (ETTm1 MSE/CRPS/KS, ETTm2 MSE) move in the fourth
#: decimal and still repeat a tick; use `"%.4f"` if those must be read off the
#: axis. `epoch_budget.csv` carries full precision.
TICK_FORMAT = "%.3f"

#: Panel-title shorthand, the same mapping the runtime figures use
#: (`runtime_analysis.ipynb::GRID_TITLE`): six panels across the text width leave
#: ~1.1 in each, which "variogram score" overruns. `METRICS` keeps the full names
#: for the coverage table and the export, where there is room. The log score
#: keeps its `/ d` -- the panel is nats per coordinate, and dropping the divisor
#: from the title would make it unreadable against the tables.
SHORT_TITLE = {"nll_per_d": r"NLL / $d$", "energy_score": "ES",
               "kernel_score": "KS", "variogram_score": "VS"}

#: Height of one data-set row, as a fraction of the text width. The curves are
#: flat, so a flat row costs no readability; 0.11 is about the floor before the
#: rotated "EUPPBench" row label runs into its neighbour's.
ROW_ASPECT = 0.11

#: The shaded band is +-1 *sample* standard deviation over the seeds. With five
#: seeds a 25/75 band would be the 2nd and 4th order statistic and jump from
#: rung to rung on noise, and a min-max envelope narrows wherever a rung is
#: missing seeds. The exported CSV carries `<metric>_std`, so figure and CSV
#: state the same quantity. Transparency comes from `pc.CONFIG.band_alpha`,
#: the setting every other band in the paper uses.
BAND_LABEL = r"$\pm 1$ s.d. over seeds"


def axis_label(task):
    dataset = task.split("/")[0]
    return AXIS_LABEL.get(dataset, dataset)


def _panel(ax, agg, task, metric, arms=ARMS):
    """One metric of one task: both arms against the budget fraction.

    Returns the number of arms drawn, so a caller can tell "this arm has not run
    yet" (a legitimate gap) from "nothing matched at all" (a bug). An empty
    panel is otherwise indistinguishable from a working one with no data, which
    is how a blank figure gets mistaken for a finished one.
    """
    drawn = 0
    for arm in arms:
        if (task, arm["key"]) not in agg.index:
            continue                      # that arm has not been trained yet
        rows = agg.loc[(task, arm["key"])]
        x = [frac for frac, _budget in rows.index]
        mean = rows[(metric, "mean")].to_numpy(dtype=float)
        sd = rows[(metric, "std")].to_numpy(dtype=float)
        if np.all(np.isnan(mean)):
            continue                      # metric not computed for this task
        # A single-seed rung has no sd: zeroing it pinches the band to the line
        # there, where a `NaN` would instead open a hole in `fill_between` and
        # take the neighbouring rungs' band with it. The marker still shows that
        # the rung ran.
        sd = np.nan_to_num(sd, nan=0.0)
        ax.plot(x, mean, color=arm["color"], marker=arm["marker"], ms=2.8,
                lw=1.1, label=arm["label"], zorder=3)
        ax.fill_between(x, mean - sd, mean + sd, color=arm["color"],
                        alpha=CFG.band_alpha, lw=0, zorder=2)
        drawn += 1
    # Six panels share a double-column width, so each is ~1.1 in and the full
    # five-rung tick set collides with itself. Three ticks still show the range
    # and the midpoint, and the rungs are visible as the markers themselves.
    ax.set_xticks([0.2, 0.6, 1.0])
    ax.margins(x=0.10)
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=3))
    # (no `ticklabel_format` here: it only accepts a ScalarFormatter, and a
    # fixed format string has no offset to switch off in the first place.)
    ax.yaxis.set_major_formatter(plt.FormatStrFormatter(TICK_FORMAT))
    return drawn


def plot_ablation(agg, metrics=METRICS, arms=ARMS, save=True, row_aspect=ROW_ASPECT):
    """The ablation figure: one row per data set, one column per metric.

    The y scale is per panel, never shared down a column: the metrics differ by
    orders of magnitude between data sets, and the figure is read as "does the
    gap between the two arms close as the budget grows", which is a within-panel
    comparison. `row_aspect` is the height of one data-set row as a fraction of the
    text width -- `ROW_ASPECT`, the paper's, unless a copy asks for taller panels.
    """
    tasks = tasks_of(agg)
    fig, axes = pc.subplots(len(tasks), len(metrics),
                            aspect=row_aspect * len(tasks), squeeze=False,
                            sharex=True)
    drawn = 0
    for row, task in enumerate(tasks):
        for col, (metric, label) in enumerate(metrics):
            ax = axes[row][col]
            drawn += _panel(ax, agg, task, metric, arms)
            if row == 0:
                ax.set_title(SHORT_TITLE.get(metric, label))
            if col == 0:
                ax.set_ylabel(axis_label(task))
    if not drawn:
        raise RuntimeError(
            f"nothing plotted -- the aggregate has {list(agg.index.names)} but "
            f"no series matched. Check that the index level `task` holds task "
            f"*names* and `arm` one of {[a['key'] for a in arms]}."
        )
    handles = [plt.Line2D([], [], color=a["color"], marker=a["marker"],
                          ms=2.8, lw=1.1, label=a["label"]) for a in arms]
    # A neutral grey patch for the band: it means the same thing on both arms,
    # so tying it to either colour would read as "this arm has a band and that
    # one does not".
    handles.append(Patch(facecolor="0.5", alpha=CFG.band_alpha,
                         edgecolor="none", label=BAND_LABEL))
    # The budget belongs on the axis it parametrises, so it is labelled under
    # the panels and the key goes below that. Constrained layout's
    # `loc="outside lower center"` would draw the legend on top of a `supxlabel`,
    # and `rect` does not help (the engine places `supxlabel` outside the rect).
    # So both are placed by hand in a strip the engine is told to keep clear:
    # the legend pinned to the bottom of the canvas, the label just above it.
    # Nothing leaves the canvas, so `pc.save` keeps its no-tight-bbox rule (a
    # tight bbox would silently change the paper width).
    label = fig.text(0.5, 0.0, "epoch budget (fraction of the configured budget)",
                     ha="center", va="bottom",
                     fontsize=plt.rcParams["figure.labelsize"])
    legend = fig.legend(handles=handles, loc="lower center",
                        bbox_to_anchor=(0.5, 0.0), bbox_transform=fig.transFigure,
                        ncol=len(handles), frameon=True, fancybox=True,
                        framealpha=1.0, borderpad=0.6, columnspacing=1.6,
                        edgecolor="0.7")
    # Measured on a first layout pass, never guessed: how much room the two need
    # follows the font size, the number of entries and the figure height, all of
    # which move. `fig.text` is not managed by the layout engine, so the label
    # stays where it is put across later draws and `savefig`.
    fig.draw_without_rendering()
    to_fig = fig.transFigure.inverted()
    legend_box = legend.get_window_extent().transformed(to_fig)
    label_box = label.get_window_extent().transformed(to_fig)
    pad = 0.012
    label.set_position((0.5, legend_box.y1 + pad))
    strip = legend_box.y1 + label_box.height + 2 * pad
    fig.get_layout_engine().set(rect=(0, strip, 1, 1 - strip))
    if save:
        print(f"wrote {pc.save(fig, 'ablation_epoch_budget.pdf')}")
    return fig


plot_ablation(agg)

### Presentation copy — four metrics, taller rows

The figure above with the same rows and construction, restricted to **MSE, CRPS, ES and VS** (the
kernel-score and log-score columns dropped), and with taller panels: `row_aspect=0.18` against
the paper's `ROW_ASPECT = 0.11`. Written to `figures/presentation/` (git-ignored), never over the paper figure.

In [ ]:
#: The four metrics the presentation copies keep, in the grid's own order.
PRESENTATION_METRICS = [m for m in METRICS
                        if m[0] in ("mse", "crps", "energy_score", "variogram_score")]

fig = plot_ablation(agg, metrics=PRESENTATION_METRICS, save=False, row_aspect=0.18)
print(f"wrote {pc.save(fig, pc.PRESENTATION_DIR / 'ablation_epoch_budget.pdf')}")
plt.show()


## Export to the results tree

The aggregate goes to `<experiment>/epoch_budget/` in the results tree, a
sibling of `tables/`, `calibration/` and `runtime_analysis/`, under the tail of
the checkpoint tree's identity path -- so an ablation file sits beside the
checkpoints and diagnostics of the models it describes. The per-seed rows travel
with it: they carry what an aggregate cannot (every seed, the split each rung
trained at, the effective warmups, the selected epoch and the wall clock).

`umask 002` keeps the written files group-writable. Set `EXPORT = True` to write.

The export **rewrites whole files** rather than appending, so repeating it is
safe and is how a rung that has gained seeds gets replaced. Keep `EXPORT` set
to `False` in the committed notebook, so a "run all" never writes the results
tree. Any prose elsewhere that quotes coverage, seed counts or result ranges is
not derived from the exported files and has to be updated by hand after a
re-export.

In [ ]:
EXPORT = False

#: Columns of an exported file, in order. The metric columns carry full
#: precision -- the figure rounds, the machine-readable record does not.
EXPORT_FIELDS = (
    ["arm", "label", "frac", "budget", "stage1_epochs", "stage2_epochs",
     "slug", "n_seeds", "d", "params", "best_epoch", "wall_s"]
    + [m for m, _ in METRICS]
    + [f"{m}_std" for m, _ in METRICS]
)


def out_dir_for(row, root=SHARED):
    """`<experiment>/epoch_budget/<dataset>[/<mode>][/T<horizon>]`.

    Built from the identity columns of any row of the task, not from the task
    name, so the path stays the tail of the checkpoint tree's own layout.
    """
    path = root / row["experiment"] / "epoch_budget" / row["dataset"]
    if row["mode"]:
        path = path / row["mode"]
    if row["horizon"]:
        path = path / f"T{int(row['horizon'])}"
    return path


def export(df, agg, root=SHARED):
    """Write the per-seed rows and one aggregated CSV per task."""
    import os

    labels = {a["key"]: a["label"] for a in ARMS}
    previous = os.umask(0o002)
    written = []
    try:
        for experiment, subset in df.groupby("experiment"):
            out_dir = root / experiment / "epoch_budget"
            out_dir.mkdir(parents=True, exist_ok=True)
            path = out_dir / "scores.jsonl"
            with open(path, "w") as fh:
                for record in subset.drop(columns=["task"]).to_dict("records"):
                    fh.write(json.dumps(record, default=str) + "\n")
            written.append(path)

        for task, subset in df.groupby("task"):
            out_dir = out_dir_for(subset.iloc[0], root)
            out_dir.mkdir(parents=True, exist_ok=True)
            path = out_dir / "epoch_budget.csv"
            rows = []
            for (arm, frac, budget), row in agg.loc[(task,)].iterrows():
                cell = subset[(subset["arm"] == arm) & (subset["frac"] == frac)]
                record = {
                    "arm": arm, "label": labels.get(arm, arm),
                    "frac": frac, "budget": budget,
                    "stage1_epochs": cell["stage1_epochs"].iloc[0],
                    "stage2_epochs": cell["stage2_epochs"].iloc[0],
                    "slug": cell["slug"].iloc[0],
                    "n_seeds": int(cell["seed"].nunique()),
                    "d": cell["d"].iloc[0],
                    "params": cell["params"].iloc[0],
                    "best_epoch": cell["best_epoch"].mean(),
                    "wall_s": cell["wall_s"].mean(),
                }
                for metric, _ in METRICS:
                    record[metric] = row[(metric, "mean")]
                    record[f"{metric}_std"] = row[(metric, "std")]
                rows.append(record)
            pd.DataFrame(rows)[EXPORT_FIELDS].to_csv(path, index=False)
            written.append(path)
    finally:
        os.umask(previous)
    return written


if EXPORT:
    for path in export(df, agg):
        print(f"wrote {path}")
else:
    print("EXPORT is False -- set it to True to write into the shared tree")